# Email Spam Classifier

This notebook demonstrates a complete end-to-end pipeline for classifying emails as Spam or Ham. We will use both classical TF-IDF features and modern Sentence-BERT embeddings.

In [ ]:
import os, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from scipy.sparse import hstack, csr_matrix

import xgboost as xgb
from sentence_transformers import SentenceTransformer
import shap
from sklearn.manifold import TSNE
import kagglehub

## Data Loading

In [ ]:
path = kagglehub.dataset_download("venky73/spam-mails-dataset")
csv_path = os.path.join(path, "spam_ham_dataset.csv")
df = pd.read_csv(csv_path)

# Drop index column
df.drop(columns=[c for c in df.columns if "Unnamed" in c], inplace=True)
print("Dataset shape:", df.shape)
display(df.head())

## EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1. Class distribution
label_counts = df["label"].value_counts()
axes[0].bar(label_counts.index, label_counts.values, color=["#4CAF50", "#F44336"])
axes[0].set_title("Class Distribution")

# 2. Email length
df["text_len"] = df["text"].str.len()
for label, color in zip(["ham", "spam"], ["#4CAF50", "#F44336"]):
    sub = df[df["label"] == label]["text_len"]
    axes[1].hist(sub, bins=60, alpha=0.6, label=label, color=color)
axes[1].set_title("Length Distribution")
axes[1].legend()
plt.show()

## Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " [URL] ", text)
    text = re.sub(r"\S+@\S+", " [EMAIL] ", text)
    text = re.sub(r"\d+", " [NUM] ", text)
    text = re.sub(r"[^a-z\s\[\]]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join([lemmatizer.lemmatize(t) for t in text.split() if t not in STOP_WORDS or t.startswith("[")])

df["clean_text"] = df["text"].apply(clean_text)

df["num_urls"] = df["text"].apply(lambda x: len(re.findall(r"http\S+|www\S+", str(x))))
df["num_caps"] = df["text"].apply(lambda x: sum(1 for c in str(x) if c.isupper()))
df["caps_ratio"] = df["num_caps"] / (df["text"].str.len() + 1)
df["num_excl"] = df["text"].str.count("!")
df["num_words"] = df["clean_text"].str.split().str.len()

## Feature Engineering

In [ ]:
X_text = df["clean_text"]
y = df["label_num"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

hc_cols = ["num_urls", "caps_ratio", "num_excl", "num_words"]
hc_train = csr_matrix(df.loc[X_train_raw.index, hc_cols].values.astype(float))
hc_test = csr_matrix(df.loc[X_test_raw.index, hc_cols].values.astype(float))

X_train_tfidf_hc = hstack([X_train_tfidf, hc_train])
X_test_tfidf_hc = hstack([X_test_tfidf, hc_test])

In [ ]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_train_emb = sbert.encode(X_train_raw.tolist(), batch_size=64, show_progress_bar=True)
X_test_emb = sbert.encode(X_test_raw.tolist(), batch_size=64, show_progress_bar=True)

X_train_combined = hstack([X_train_tfidf_hc, csr_matrix(X_train_emb)])
X_test_combined = hstack([X_test_tfidf_hc, csr_matrix(X_test_emb)])

## Model Training

In [ ]:
models = {
    "Logistic Regression (TF-IDF)": (LogisticRegression(max_iter=1000, C=1.0), X_train_tfidf, X_test_tfidf),
    "Logistic Regression (Combined)": (LogisticRegression(max_iter=1000, C=1.0), X_train_combined, X_test_combined)
}

results = {}
for name, (model, X_tr, X_te) in models.items():
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_te)
    
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    results[name] = {"F1": f1, "AUC": auc, "model": model, "preds": preds}
    print(f"{name} -> F1: {f1:.4f}, AUC: {auc:.4f}")

## Analysis with SHAP and t-SNE

In [ ]:
lr_model = results["Logistic Regression (TF-IDF)"]["model"]
bg_sample = X_train_tfidf[:200]
explainer = shap.LinearExplainer(lr_model, bg_sample, feature_perturbation="interventional")
shap_vals = explainer.shap_values(X_test_tfidf[:200])

shap.summary_plot(shap_vals, X_test_tfidf[:200], feature_names=tfidf.get_feature_names_out())

In [ ]:
tsne = TSNE(n_components=2, perplexity=40, max_iter=1000, random_state=42)
emb_2d = tsne.fit_transform(X_train_emb[:1000])
labels_sub = y_train.values[:1000]

plt.figure(figsize=(8, 6))
for cls, color, name_cls in [(0, "#43A047", "Ham"), (1, "#E53935", "Spam")]:
    mask = labels_sub == cls
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, label=name_cls, alpha=0.6, s=20)
plt.title("t-SNE of SBERT Embeddings")
plt.legend()
plt.show()